# F.R.E.S.H — Complete AI Training Notebook

Notebook ini dibuat untuk project **F.R.E.S.H (Food Resource Efficiency & Smart Handling)**.

Target:
1. Load dataset sementara.
2. Data cleaning dan EDA.
3. Feature engineering untuk prediksi risiko food waste.
4. Training model baseline Scikit-learn.
5. Opsional training model TensorFlow/Keras.
6. Export model untuk FastAPI.
7. Contoh inference payload yang dipakai frontend/backend.

> Jalankan notebook ini di Google Colab. Upload 3 CSV ke folder `/content/data/` atau ubah path sesuai lokasi file.

In [ ]:
# Jika di Google Colab, jalankan install berikut
!pip -q install pandas numpy matplotlib scikit-learn joblib fastapi uvicorn

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, datetime

DATA_DIR = "/content/data"  # ubah ke "." jika file CSV berada di direktori notebook lokal
os.makedirs(DATA_DIR, exist_ok=True)

DAIRY_PATH = f"{DATA_DIR}/cleaned_dairy_dataset.csv"
FRUIT_PATH = f"{DATA_DIR}/cleaned_fruits_dataset.csv"
WASTAGE_PATH = f"{DATA_DIR}/cleaned_food_wastage_data.csv"

## 1. Load Dataset

In [ ]:
dairy = pd.read_csv(DAIRY_PATH)
fruits = pd.read_csv(FRUIT_PATH)
wastage = pd.read_csv(WASTAGE_PATH)

print("Dairy:", dairy.shape)
display(dairy.head())

print("Fruits:", fruits.shape)
display(fruits.head())

print("Wastage:", wastage.shape)
display(wastage.head())

## 2. Data Understanding dan Missing Value

In [ ]:
for name, df in [("dairy", dairy), ("fruits", fruits), ("wastage", wastage)]:
    print("\n==", name.upper(), "==")
    print(df.info())
    print(df.isna().sum().sort_values(ascending=False).head(10))

## 3. Feature Engineering

Fitur utama MVP: `category`, `quantity`, `shelf_life`, `storage_condition`, dan `days_to_expiry`.

In [ ]:
def label_from_priority(x):
    s = str(x).lower()
    if "tinggi" in s or "high" in s:
        return "High Risk"
    if "sedang" in s or "medium" in s:
        return "Warning"
    return "Safe"

# Dairy dataset
dfd = dairy.copy()
dfd["current_date"] = pd.to_datetime(dfd["Date"], errors="coerce")
dfd["exp_date"] = pd.to_datetime(dfd["expiration_date"], errors="coerce")
dfd["days_to_expiry"] = (dfd["exp_date"] - dfd["current_date"]).dt.days
dfd["category"] = "Dairy"
dfd["quantity"] = pd.to_numeric(dfd["Quantity in Stock (liters/kg)"], errors="coerce").fillna(
    pd.to_numeric(dfd["quantity"], errors="coerce")
)
dfd["shelf_life"] = pd.to_numeric(dfd["shelf_life"], errors="coerce")
dfd["storage_condition"] = dfd["storage_condition"].astype(str)
dfd["risk_level"] = dfd["notification_priority"].apply(label_from_priority)

train_dairy = dfd[["category","quantity","shelf_life","storage_condition","days_to_expiry","risk_level"]].dropna()
train_dairy.head()

In [ ]:
# Fruits dataset: karena belum punya transaksi inventory per user, kita buat data simulasi realistis dari shelf_life.
rng = np.random.default_rng(42)
rows = []

for _, r in fruits.iterrows():
    shelf = int(r["shelf_life"])
    for _ in range(8):
        days = int(rng.integers(-2, max(shelf + 1, 2)))
        ratio = days / max(shelf, 1)
        if days <= 1 or ratio <= 0.15:
            risk = "High Risk"
        elif days <= 3 or ratio <= 0.35:
            risk = "Warning"
        else:
            risk = "Safe"
        rows.append({
            "category": "Fruit",
            "quantity": float(rng.integers(1, 10)),
            "shelf_life": shelf,
            "storage_condition": "Refrigerated" if shelf < 10 else "Room Temperature",
            "days_to_expiry": days,
            "risk_level": risk
        })

train_fruit = pd.DataFrame(rows)
train_fruit.head()

In [ ]:
# Food wastage event dataset: digunakan sebagai tambahan pola umum.
fwe = wastage.copy()
fwe["category"] = fwe["type_of_food"].astype(str)
fwe["quantity"] = pd.to_numeric(fwe["quantity_of_food"], errors="coerce")
fwe["shelf_life"] = 7
fwe["storage_condition"] = fwe["storage_conditions"].astype(str)

q33 = fwe["wastage_food_amount"].quantile(0.33)
q66 = fwe["wastage_food_amount"].quantile(0.66)

fwe["days_to_expiry"] = np.where(fwe["wastage_food_amount"] > q66, 1,
                         np.where(fwe["wastage_food_amount"] > q33, 3, 7))
fwe["risk_level"] = np.where(fwe["wastage_food_amount"] > q66, "High Risk",
                     np.where(fwe["wastage_food_amount"] > q33, "Warning", "Safe"))

train_wastage = fwe[["category","quantity","shelf_life","storage_condition","days_to_expiry","risk_level"]].dropna()
train_wastage.head()

In [ ]:
train = pd.concat([train_dairy, train_fruit, train_wastage], ignore_index=True)
train["quantity"] = pd.to_numeric(train["quantity"], errors="coerce").fillna(1).clip(lower=0)
train["shelf_life"] = pd.to_numeric(train["shelf_life"], errors="coerce").fillna(7).clip(lower=1)
train["days_to_expiry"] = pd.to_numeric(train["days_to_expiry"], errors="coerce").fillna(7)

print(train.shape)
display(train.head())
display(train["risk_level"].value_counts())

## 4. EDA Sederhana

In [ ]:
train["risk_level"].value_counts().plot(kind="bar")
plt.title("Distribusi Label Risiko")
plt.xlabel("Risk Level")
plt.ylabel("Jumlah")
plt.show()

train.groupby("risk_level")["days_to_expiry"].describe()

In [ ]:
train.groupby("category")["risk_level"].value_counts().unstack(fill_value=0).plot(kind="bar", stacked=True)
plt.title("Risiko Berdasarkan Kategori")
plt.xlabel("Kategori")
plt.ylabel("Jumlah")
plt.show()

## 5. Training Model Baseline Scikit-learn

Model ini ringan dan cocok untuk API MVP di Railway.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

features = ["category", "quantity", "shelf_life", "storage_condition", "days_to_expiry"]
X = train[features]
y = train["risk_level"]

label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), ["quantity", "shelf_life", "days_to_expiry"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["category", "storage_condition"])
])

model = RandomForestClassifier(
    n_estimators=120,
    random_state=42,
    class_weight="balanced",
    max_depth=8
)

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=label_encoder.classes_))

## 6. Export Model untuk FastAPI

In [ ]:
ARTIFACT_DIR = "/content/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

joblib.dump(
    {"pipeline": pipeline, "label_encoder": label_encoder},
    f"{ARTIFACT_DIR}/risk_model.joblib"
)

metadata = {
    "features": features,
    "classes": label_encoder.classes_.tolist(),
    "accuracy": float(accuracy_score(y_test, pred)),
    "trained_at": datetime.utcnow().isoformat() + "Z",
    "training_rows": int(len(train))
}

with open(f"{ARTIFACT_DIR}/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

metadata

## 7. Opsional: TensorFlow/Keras Deep Learning

Project plan meminta TensorFlow/Keras dan komponen custom. Bagian ini opsional karena model TensorFlow lebih berat untuk deployment gratis. Untuk memenuhi learning path AI, jalankan bagian ini di Colab.

In [ ]:
# Optional install:
# !pip -q install tensorflow

try:
    import tensorflow as tf
    from tensorflow.keras import layers

    X_processed = preprocess.fit_transform(X).toarray()
    y_onehot = tf.keras.utils.to_categorical(y_enc)

    Xtr, Xte, ytr, yte = train_test_split(
        X_processed, y_onehot, test_size=0.2, random_state=42, stratify=y_enc
    )

    class CustomDenseBlock(layers.Layer):
        def __init__(self, units):
            super().__init__()
            self.dense = layers.Dense(units, activation="relu")
            self.dropout = layers.Dropout(0.2)

        def call(self, inputs, training=False):
            x = self.dense(inputs)
            return self.dropout(x, training=training)

    inputs = tf.keras.Input(shape=(Xtr.shape[1],))
    x = CustomDenseBlock(64)(inputs)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(y_onehot.shape[1], activation="softmax")(x)

    dl_model = tf.keras.Model(inputs, outputs)
    dl_model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = dl_model.fit(
        Xtr, ytr,
        validation_split=0.2,
        epochs=30,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    dl_model.save(f"{ARTIFACT_DIR}/risk_model.keras")
    joblib.dump(preprocess, f"{ARTIFACT_DIR}/tf_preprocessor.joblib")
    joblib.dump(label_encoder, f"{ARTIFACT_DIR}/tf_label_encoder.joblib")

except Exception as e:
    print("TensorFlow section skipped/error:", e)

## 8. Test Inference

In [ ]:
sample_payload = pd.DataFrame([{
    "category": "Dairy",
    "quantity": 1,
    "shelf_life": 7,
    "storage_condition": "Refrigerated",
    "days_to_expiry": 2
}])

pred_idx = pipeline.predict(sample_payload)[0]
proba = pipeline.predict_proba(sample_payload)[0]
risk = label_encoder.inverse_transform([pred_idx])[0]

{
    "input": sample_payload.to_dict(orient="records")[0],
    "risk_level": risk,
    "confidence": float(proba[pred_idx])
}

## 9. Contoh Payload API FastAPI

In [ ]:
# Request ke endpoint POST /predict-risk
payload = {
    "food_name": "Susu",
    "category": "Dairy",
    "quantity": 1,
    "unit": "liter",
    "purchase_date": "2026-05-01",
    "expiration_date": "2026-05-12",
    "storage_condition": "Refrigerated",
    "shelf_life": 7
}
payload

## 10. Langkah setelah notebook selesai

1. Download folder `/content/artifacts`.
2. Copy `risk_model.joblib` dan `model_metadata.json` ke folder `backend/artifacts/`.
3. Jalankan backend FastAPI.
4. Hubungkan frontend React ke URL backend menggunakan `VITE_API_BASE_URL`.